In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [22]:
# 일반 Dataset은 추상클래스임으로 추가 작성이 필요하지만, 
# 적은 데이터와 실시간 처리 목적이 아니므로 토치 내부에서 제공하는 TensorDataset을 사용
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

In [23]:
x_train  =  torch.FloatTensor([[73,  80,  75], 
                               [93,  88,  93], 
                               [89,  91,  90], 
                               [96,  98,  100],   
                               [73,  66,  70]])  
y_train  =  torch.FloatTensor([[152],  [185],  [180],  [196],  [142]])

In [24]:
dataset = TensorDataset(x_train, y_train) # Dataset은 데이터의 위치(경로)만 기억하고 있다가, 모델이 필요로 하는 그 순간(배치 단위)에만 디스크에서 불러와 텐서로 변환
print(dataset.tensors)

(tensor([[ 73.,  80.,  75.],
        [ 93.,  88.,  93.],
        [ 89.,  91.,  90.],
        [ 96.,  98., 100.],
        [ 73.,  66.,  70.]]), tensor([[152.],
        [185.],
        [180.],
        [196.],
        [142.]]))


In [25]:
dataloader = DataLoader(dataset, batch_size=2, shuffle=True) # 배치 사이즈의 크기 == 2^n, shuffle은 epoch만큼 데이터를 섞어줌(Overfitting 방지) 

In [26]:
model = nn.Linear(3,1)
optimizer = torch.optim.SGD(model.parameters(), lr=1e-5) 

In [27]:
nb_epochs = 20
for epoch in range(nb_epochs + 1):
  for batch_idx, samples in enumerate(dataloader):
    print(batch_idx)
    print(samples)
    x_train, y_train = samples
    
    prediction = model(x_train)

    cost = F.mse_loss(prediction, y_train)

    optimizer.zero_grad()
    cost.backward()
    optimizer.step()

    print('Epoch {:4d}/{} Batch {}/{} Cost: {:.6f}'.format(
        epoch, nb_epochs, batch_idx+1, len(dataloader),
        cost.item()
        ))


0
[tensor([[93., 88., 93.],
        [73., 80., 75.]]), tensor([[185.],
        [152.]])]
Epoch    0/20 Batch 1/3 Cost: 7790.409180
1
[tensor([[73., 66., 70.],
        [89., 91., 90.]]), tensor([[142.],
        [180.]])]
Epoch    0/20 Batch 2/3 Cost: 2367.920654
2
[tensor([[ 96.,  98., 100.]]), tensor([[196.]])]
Epoch    0/20 Batch 3/3 Cost: 1192.979492
0
[tensor([[ 96.,  98., 100.],
        [ 73.,  80.,  75.]]), tensor([[196.],
        [152.]])]
Epoch    1/20 Batch 1/3 Cost: 181.853851
1
[tensor([[89., 91., 90.],
        [93., 88., 93.]]), tensor([[180.],
        [185.]])]
Epoch    1/20 Batch 2/3 Cost: 71.073486
2
[tensor([[73., 66., 70.]]), tensor([[142.]])]
Epoch    1/20 Batch 3/3 Cost: 23.581949
0
[tensor([[ 73.,  80.,  75.],
        [ 96.,  98., 100.]]), tensor([[152.],
        [196.]])]
Epoch    2/20 Batch 1/3 Cost: 2.416663
1
[tensor([[89., 91., 90.],
        [73., 66., 70.]]), tensor([[180.],
        [142.]])]
Epoch    2/20 Batch 2/3 Cost: 4.606380
2
[tensor([[93., 88., 93.]]), 

In [28]:
new_var =  torch.FloatTensor([[73, 80, 75]]) 
pred_y = model(new_var) 
print("훈련 후 입력이 73, 80, 75일 때의 예측값 :", pred_y) 

훈련 후 입력이 73, 80, 75일 때의 예측값 : tensor([[152.3486]], grad_fn=<AddmmBackward0>)


In [29]:
from torch.utils.data import Dataset

In [30]:
class CustomDataset(Dataset): 
  def __init__(self):
    self.x_data = [[73, 80, 75],
                   [93, 88, 93],
                   [89, 91, 90],
                   [96, 98, 100],
                   [73, 66, 70]]
    self.y_data = [[152], [185], [180], [196], [142]]

  def __len__(self): 
    return len(self.x_data)

  def __getitem__(self, idx): 
    x = torch.FloatTensor(self.x_data[idx])
    y = torch.FloatTensor(self.y_data[idx])
    return x, y

In [34]:
dataset = CustomDataset()
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

# x_train 데이터의 총 샘플 개수가 5개, batch_size=2로 설정
# 마지막 Batch 3/3 시 나누어 떨어지지 않아 데이터 하나 즉, 남은 짜투리만 보냄
# 따라서 Cost가 뛰게 됨. 이를 보안하기 위해 drop_last를 설정(남은 짜투리를 버리는 옵션)
# dataloader = DataLoader(dataset, batch_size=2, shuffle=True, drop_last=True)

In [35]:
model = torch.nn.Linear(3,1)
optimizer = torch.optim.SGD(model.parameters(), lr=1e-5) 

In [36]:
nb_epochs = 20
for epoch in range(nb_epochs + 1):
  for batch_idx, samples in enumerate(dataloader):
    x_train, y_train = samples

    prediction = model(x_train)

    cost = F.mse_loss(prediction, y_train)

    optimizer.zero_grad()
    cost.backward()
    optimizer.step()

    print('Epoch {:4d}/{} Batch {}/{} Cost: {:.6f}'.format(
        epoch, nb_epochs, batch_idx+1, len(dataloader),
        cost.item()
        ))

Epoch    0/20 Batch 1/3 Cost: 25960.492188
Epoch    0/20 Batch 2/3 Cost: 4419.636719
Epoch    0/20 Batch 3/3 Cost: 1476.282104
Epoch    1/20 Batch 1/3 Cost: 724.560181
Epoch    1/20 Batch 2/3 Cost: 215.571930
Epoch    1/20 Batch 3/3 Cost: 82.844551
Epoch    2/20 Batch 1/3 Cost: 18.458778
Epoch    2/20 Batch 2/3 Cost: 11.050587
Epoch    2/20 Batch 3/3 Cost: 2.398916
Epoch    3/20 Batch 1/3 Cost: 3.188336
Epoch    3/20 Batch 2/3 Cost: 0.009900
Epoch    3/20 Batch 3/3 Cost: 2.345334
Epoch    4/20 Batch 1/3 Cost: 2.232376
Epoch    4/20 Batch 2/3 Cost: 0.319289
Epoch    4/20 Batch 3/3 Cost: 1.834608
Epoch    5/20 Batch 1/3 Cost: 1.563792
Epoch    5/20 Batch 2/3 Cost: 1.248358
Epoch    5/20 Batch 3/3 Cost: 1.133710
Epoch    6/20 Batch 1/3 Cost: 0.256819
Epoch    6/20 Batch 2/3 Cost: 2.231910
Epoch    6/20 Batch 3/3 Cost: 1.275635
Epoch    7/20 Batch 1/3 Cost: 2.446507
Epoch    7/20 Batch 2/3 Cost: 0.553640
Epoch    7/20 Batch 3/3 Cost: 0.157503
Epoch    8/20 Batch 1/3 Cost: 1.331767
Epoch   

In [37]:
new_var =  torch.FloatTensor([[73, 80, 75]]) 
pred_y = model(new_var) 
print("훈련 후 입력이 73, 80, 75일 때의 예측값 :", pred_y)

훈련 후 입력이 73, 80, 75일 때의 예측값 : tensor([[150.6110]], grad_fn=<AddmmBackward0>)
